# 두 데이터의 구매 행태 비교 — Colab 실행용

위에서 아래로 실행하세요. **GPU 불필요 / H&M은 고용량 RAM 권장**.
기존 실험의 Drive 데이터 경로를 반영했습니다. 경로 수정 없이 전체 실행하고 Drive 연결을 승인하면 됩니다. 추가 Python 파일 업로드는 필요 없습니다.

- (a)~(c): 마지막 주 이전 전체 학습 자료. CLV의 최근 365일 창과 구별합니다.
- (d): 학습에 등장한 고객·상품 범위의 테스트 주 고유 구매 쌍 구성입니다. 추천 성능 평가가 아닙니다.
- PNG·SVG·집계 JSON을 로컬 `/content`에 저장한 뒤 ZIP으로 내려받습니다. 논문·원본 자료·기존 Drive 결과는 수정하지 않습니다.


## 1. 패키지와 한글 글꼴 준비

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pandas', 'numpy', 'pyarrow', 'cairosvg'], check=True)
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum', 'libcairo2'], check=True)
subprocess.run(['fc-cache', '-f'], check=True)
print('패키지·한글 글꼴 준비 완료')


## 2. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. 기존 실험 데이터 연결 확인
`lightgcn_clv_v3.py`의 SCHEMA 설정과 원격 main의 경로를 확인했습니다. H&M은 Parquet, dunnhumby는 CSV를 사용합니다. 수정 없이 실행하세요.

In [ ]:
from pathlib import Path
HM_PATH = Path('/content/drive/MyDrive/논문/data/raw/hm/transactions_train.parquet')
DH_PATH = Path('/content/drive/MyDrive/논문/data/raw/dunnhumby/transaction_data.csv')
OUT = Path('/content/purchase_behavior_out')
for path in [HM_PATH, DH_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'기존 실험 데이터에 접근할 수 없습니다: {path}. 실험에 사용한 Google 계정으로 Drive가 연결되었는지 확인하세요.')
    print(path, f'{path.stat().st_size / 1024**2:,.1f} MB')
OUT.mkdir(parents=True, exist_ok=True)


## 4. 집계·그림 함수
기존 로컬 코드와 같은 계산식입니다. 전체 열의 완전중복을 제거합니다. 금액 분포는 양수인 고객별 평균 거래금액을 데이터별 중앙값으로 나눕니다. 두 데이터의 관찰 기간 길이는 다르므로 거래량 차이를 업종 효과만으로 해석하지 않습니다.

In [ ]:
"""두 데이터의 구매 행태 비교. 모형 학습/평가 및 논문 파일 수정 없음.

실행: python purchase_behavior_compare.py --root /경로/논문준비
필수: pandas, numpy. PNG는 CairoSVG 또는 Node.js + sharp로 생성.
출력: SVG(벡터), PNG(가능할 때), 집계 JSON. 기존 EDA 이미지는 보존.
(a)~(c): 전체 학습 기간. CLV의 최근 365일 관찰 창과 구별한다.
(d): 마지막 주의 학습 고객·상품 고유 쌍. 재구매 제외 전 구성 통계이며
     재구매 추천 평가나 추천 성능 지표가 아니다.
"""
import argparse
import gc
import html
import json
import os
from pathlib import Path
import shutil
import subprocess

import numpy as np
import pandas as pd


def cdf(values):
    x, n = np.unique(values, return_counts=True)
    if not len(x):
        raise ValueError("분포를 계산할 유효 관측값이 없습니다")
    y = np.cumsum(n) / n.sum() * 100
    # 누적비율 기준 최대 0.1%p 간격으로 정점을 줄인다. 범위 절단은 없다.
    idx = np.unique(np.r_[0, np.searchsorted(y, np.linspace(0, 100, 1001)), len(x)-1])
    idx = idx[idx < len(x)]
    return {"x": x[idx].tolist(), "y": y[idx].tolist(), "n": int(n.sum())}


def summarize(tx, cut, end, name):
    if tx.empty or tx[["u", "i", "b", "t", "amount"]].isna().any().any():
        raise ValueError(f"{name}: 비어 있거나 필수 값이 누락된 자료")
    if tx.t.max() != end:
        raise ValueError(f"{name}: 자료 마지막 시점 {tx.t.max()}이 예상 {end}와 다릅니다")
    train = tx[tx.t < cut]
    test = tx[(tx.t >= cut) & (tx.t <= end)]
    baskets = train.groupby(["u", "b"], observed=True, sort=False).amount.sum()
    units = baskets.groupby(level=0, observed=True).size().to_numpy()
    value = baskets.groupby(level=0, observed=True).mean().to_numpy()
    positive = value[value > 0]
    median = float(np.median(positive))
    if not np.isfinite(median) or median <= 0:
        raise ValueError(f"{name}: 거래당 금액 중앙값이 양수가 아닙니다")
    pairs = train[["u", "i"]].drop_duplicates()
    popularity = pairs.groupby("i", observed=True).size().sort_values(ascending=False).to_numpy()
    concentration = {"x": np.r_[0, np.arange(1, len(popularity)+1)/len(popularity)*100].tolist(),
                     "y": np.r_[0, np.cumsum(popularity)/popularity.sum()*100].tolist()}
    # 전체 곡선의 균등 상품 순위 표본. 마지막 점은 반드시 보존한다.
    idx = np.unique(np.linspace(0, len(popularity), min(len(popularity)+1, 1001), dtype=int))
    concentration = {k: [v[j] for j in idx] for k, v in concentration.items()}
    all_test_pairs = test[["u", "i"]].drop_duplicates()
    eligible = all_test_pairs[all_test_pairs.u.isin(pairs.u.unique()) &
                              all_test_pairs.i.isin(pairs.i.unique())]
    keys = pd.MultiIndex.from_frame(pairs)
    repeated = int(pd.MultiIndex.from_frame(eligible).isin(keys).sum())
    total = len(eligible)
    if not total:
        raise ValueError(f"{name}: 학습 고객·상품 범위의 테스트 구매 쌍이 없습니다")
    result = dict(name=name, train_rows=len(train), train_customers=len(units),
                  train_items=len(popularity), train_pairs=len(pairs),
                  median_units=float(np.median(units)), units=cdf(units),
                  median_customer_mean_basket_amount=median,
                  nonpositive_customer_mean_amount=int((value <= 0).sum()),
                  value=cdf(positive / median), concentration=concentration,
                  test_all_pairs=len(all_test_pairs), test_eligible_pairs=total,
                  test_excluded_pairs=len(all_test_pairs)-total,
                  test_repeat_pairs=repeated, test_new_pairs=total-repeated,
                  test_new_pct=100*(total-repeated)/total)
    assert result["test_repeat_pairs"] + result["test_new_pairs"] == total
    assert abs(result["concentration"]["y"][-1]-100) < 1e-8
    print(name, "학습 고객", len(units), "고객 미구매 쌍 비율", round(result["test_new_pct"], 2), flush=True)
    return result


def read_data(root, dataset, source=None):
    if dataset == "hm":
        path = Path(source) if source else root / "data/hm/transactions_train.csv"
        raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path, dtype={"customer_id": "category", "t_dat": "category",
                                      "article_id": "int32", "price": "float64",
                                      "sales_channel_id": "int8"})
        before = len(raw)
        raw = raw.drop_duplicates()
        raw["customer_id"] = raw["customer_id"].astype("category")
        raw["t_dat"] = raw["t_dat"].astype("category")
        days = (pd.to_datetime(raw.t_dat.cat.categories)-pd.Timestamp("2018-09-20")).days.to_numpy()
        t = days[raw.t_dat.cat.codes].astype("int16")
        tx = pd.DataFrame(dict(u=raw.customer_id.cat.codes.to_numpy(),
                               i=raw.article_id.to_numpy(), t=t, b=t,
                               amount=raw.price.to_numpy()))
        cut = (pd.Timestamp("2020-09-16")-pd.Timestamp("2018-09-20")).days
        end = (pd.Timestamp("2020-09-22")-pd.Timestamp("2018-09-20")).days
        name = "H&M"
    else:
        path = Path(source) if source else root / "data/dunnhumby/dunnhumby_The-Complete-Journey CSV/transaction_data.csv"
        # 전체 열로 완전중복을 판정해 금액이 다른 구매행은 보존한다.
        raw = pd.read_csv(path)
        before = len(raw)
        raw = raw.drop_duplicates()
        tx = raw.rename(columns={"household_key": "u", "PRODUCT_ID": "i",
                                 "DAY": "t", "BASKET_ID": "b", "SALES_VALUE": "amount"})[
                                     ["u", "i", "t", "b", "amount"]].copy()
        cut, end, name = 705, 711, "dunnhumby"
    result = summarize(tx, cut, end, name)
    result.update(source=str(path.resolve()), source_bytes=path.stat().st_size,
                  source_mtime_ns=path.stat().st_mtime_ns,
                  raw_rows=before, exact_duplicate_rows_removed=before-len(raw),
                  test_start="2020-09-16" if dataset == "hm" else "DAY705",
                  test_end="2020-09-22" if dataset == "hm" else "DAY711")
    return result


def render(data, out):
    svg = []
    def text(x, y, label, size=27, anchor="middle", extra=""):
        svg.append(f'<text x="{x}" y="{y}" font-size="{size}" text-anchor="{anchor}" {extra}>{html.escape(str(label))}</text>')
    def line(x1, y1, x2, y2, color="#ddd", width=1.5):
        svg.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{color}" stroke-width="{width}"/>')
    panels = [(150, 170, 880, 450), (1350, 170, 880, 450),
              (150, 970, 880, 450), (1350, 970, 880, 450)]
    titles = ["(a) 고객별 거래 횟수", "(b) 고객별 거래당 평균 구매 금액",
              "(c) 상품 구매의 집중도", "(d) 고객 미구매 상품 구매 비중"]
    xlabs = ["거래 단위 수 (로그 눈금)", "각 데이터 중앙값 대비 배수 (로그 눈금)",
             "구매 고객 수 상위 상품의 누적 비율 (%)", "학습에 등장한 고객·상품 범위"]
    ylabs = ["누적 고객 비율 (%)", "누적 고객 비율 (%)", "누적 고객–상품 고유 쌍 비율 (%)", "테스트 주 고유 구매 쌍 비율 (%)"]
    for j, (x,y,w,h) in enumerate(panels):
        text(x+w/2, y-85, titles[j], 36)
        for v in range(0,101,20):
            yy=y+h-h*v/100; line(x,yy,x+w,yy); text(x-20,yy+9,v,25,"end")
        line(x,y,x,y+h,"#222",2); line(x,y+h,x+w,y+h,"#222",2)
        text(x+w/2,y+h+96,xlabs[j],28)
        text(x-95,y+h/2,ylabs[j],27,extra=f'transform="rotate(-90 {x-95} {y+h/2})"')
        if j < 3:
            for k, d in enumerate(data.values()):
                xx=x+180+k*340
                dash='stroke-dasharray="12 8"' if k==0 else ''
                color=["#888","#222"][k]
                svg.append(f'<line x1="{xx}" y1="{y-30}" x2="{xx+75}" y2="{y-30}" stroke="{color}" stroke-width="4" {dash}/>')
                text(xx+90,y-20,d["name"],27,"start")
            key=["units","value","concentration"][j]
            lo = 1 if j==0 else min(d[key]["x"][0] for d in data.values()) if j==1 else 0
            hi = max(d[key]["x"][-1] for d in data.values()) if j<2 else 100
            def xp(v):
                return x+w*((np.log(v)-np.log(lo))/(np.log(hi)-np.log(lo)) if j<2 else v/100)
            ticks = [1,10,100,1000] if j==0 else [0.01,0.1,1,10,100] if j==1 else [0,20,40,60,80,100]
            for v in ticks:
                if lo<=v<=hi:
                    line(xp(v),y+h,xp(v),y+h+8,"#222"); text(xp(v),y+h+40,f"{v:g}",25)
            for k,d in enumerate(data.values()):
                a=d[key]; points=" ".join(f"{xp(v):.2f},{y+h-h*p/100:.2f}" for v,p in zip(a["x"],a["y"]))
                dash='stroke-dasharray="12 8"' if k==0 else ''
                svg.append(f'<polyline points="{points}" fill="none" stroke="{["#888","#222"][k]}" stroke-width="4" {dash}/>')
        else:
            for k,d in enumerate(data.values()):
                xx=x+w*(k+.5)/2; p=d["test_new_pct"]; hh=h*p/100
                svg.append(f'<rect x="{xx-110}" y="{y+h-hh}" width="220" height="{hh}" fill="{["#bbb","#333"][k]}" stroke="#222"/>')
                text(xx,y+h-hh-18,f"{p:.1f}%",31); text(xx,y+h+42,d["name"],30)
    text(1200,1610,"주: (a)~(c)는 마지막 주를 제외한 전체 학습 기간. 거래 단위는 H&M의 고객별 구매일, dunnhumby의 장바구니.",26)
    text(1200,1655,"(b)는 고객별 거래당 평균 구매 금액을 각 데이터의 양수 값 중앙값으로 나눈 분포이며, 절대 금액 비교가 아니다.",26)
    text(1200,1700,"(d)는 마지막 주의 고유 구매 쌍 중 학습에서 구매하지 않은 쌍의 비율. 추천 성능이나 재구매 추천 평가가 아니다.",26)
    result='<svg xmlns="http://www.w3.org/2000/svg" width="2400" height="1760"><rect width="2400" height="1760" fill="white"/><g font-family="Apple SD Gothic Neo, NanumGothic, sans-serif" fill="#111">'+''.join(svg)+'</g></svg>'
    target=out/"purchase_behavior_compare.svg"
    target.write_text(result,encoding="utf-8")
    # 벡터 SVG는 항상 생성한다. PNG는 설치된 렌더러만 사용한다.
    runtime=Path.home()/".cache/codex-runtimes/codex-primary-runtime/dependencies/node"
    node=shutil.which("node") or str(runtime/"bin/node")
    sharp=os.environ.get("SHARP_MODULE",str(runtime/"node_modules/sharp"))
    if Path(node).exists() and Path(sharp).exists():
        subprocess.run([node,"-e","require(process.argv[1])(process.argv[2]).png().toFile(process.argv[3]).catch(e=>{console.error(e);process.exit(1)})",sharp,str(target),str(target.with_suffix(".png"))],check=True)
    else:
        import cairosvg
        cairosvg.svg2png(url=str(target), write_to=str(target.with_suffix(".png")))


def self_check():
    tx=pd.DataFrame([(1,10,1,1,2.),(1,20,1,1,4.),(1,10,2,2,6.),
                     (2,20,3,2,3.),(1,10,4,3,2.),(2,10,5,3,2.),
                     (1,99,4,3,2.),(9,10,6,3,2.)],columns=["u","i","b","t","amount"])
    d=summarize(tx,3,3,"self-check")
    assert d["train_pairs"]==3 and d["test_eligible_pairs"]==2
    assert d["test_new_pairs"]==1 and d["test_new_pct"]==50
    assert d["test_excluded_pairs"]==2
    assert d["median_customer_mean_basket_amount"]==4.5
    assert np.allclose(cdf([1,1,2])["y"],[100*2/3,100.])




## 5. 실제 데이터 집계 및 그림 생성
H&M Parquet를 읽고 중복을 제거하는 동안 시간이 걸릴 수 있습니다. 메모리가 부족하면 런타임을 고용량 RAM으로 바꿔 처음부터 실행하세요. 기존 학습모형을 불러오거나 평가하지 않습니다.

In [ ]:
self_check()
data = {}
for dataset, source_path in [('hm', HM_PATH), ('dh', DH_PATH)]:
    print('집계 시작:', dataset, flush=True)
    data[dataset] = read_data(Path('/content'), dataset, source=source_path)
    gc.collect()
(OUT / 'aggregates.json').write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
render(data, OUT)
from IPython.display import display, Image
display(Image(filename=str(OUT / 'purchase_behavior_compare.png'), width=1100))
display(pd.DataFrame([{
    '데이터': d['name'], '학습 고객': d['train_customers'],
    '학습 상품': d['train_items'], '테스트 범위 내 고유쌍': d['test_eligible_pairs'],
    '고객 미구매쌍': d['test_new_pairs'], '미구매쌍 비율(%)': d['test_new_pct']
} for d in data.values()]))


## 6. 결과 내려받기
ZIP에 PNG, 벡터 SVG, 집계 근거 JSON을 포함합니다. Drive 저장이 필요하면 내려받은 파일을 원하는 폴더에 보관하세요.

In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/purchase_behavior_results', 'zip', root_dir=OUT)
files.download(archive)
